In [7]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [8]:
load_dotenv()
url = os.environ.get("SUPABASE_URL").strip()
key = os.environ.get("SUPABASE_KEY").strip()

In [9]:
supabase: Client = create_client(url, key)
response = supabase.table('matches').select('*').limit(10000).execute()
df_models = pd.DataFrame(response.data)

print(f"success: downloaded {len(df_models)} records")

success: downloaded 8445 records


In [10]:
df_models = df_models.sort_values('date').reset_index(drop=True)

In [11]:
features = [
    'b365h', 'b365d', 'b365a',
    'home_team_goals_avg_last_5', 'away_goals_avg_last_5', 
    'home_goals_conceded_avg_last_5', 'away_goals_conceded_avg_last_5',
    'ht_home_goals_avg_last_5', 'ht_away_goals_avg_last_5',
    'home_form_1h_last_5', 'away_form_1h_last_5', 'home_form_2h_last_5', 'away_form_2h_last_5',
    'home_shots_avg_last_5', 'away_shots_avg_last_5', 
    'home_shots_target_avg_last_5', 'away_shots_target_avg_last_5',
    'home_red_cards_avg_last_5', 'away_red_cards_avg_last_5',
    'home_points_avg_last_5', 'away_points_avg_last_5', 
    'home_overall_points_last_5', 'away_overall_points_last_5'
]



In [12]:
missing_values = df_models.isnull().mean() * 100
missing_values[missing_values > 0]

home_team_goals_avg_last_5        1.539372
away_goals_avg_last_5             1.539372
ht_home_goals_avg_last_5          1.539372
ht_away_goals_avg_last_5          1.539372
home_form_1h_last_5               1.539372
away_form_1h_last_5               1.539372
home_form_2h_last_5               1.539372
away_form_2h_last_5               1.539372
home_shots_avg_last_5             1.539372
away_shots_avg_last_5             1.539372
home_shots_target_avg_last_5      1.539372
away_shots_target_avg_last_5      1.539372
home_red_cards_avg_last_5         1.539372
away_red_cards_avg_last_5         1.539372
home_goals_conceded_avg_last_5    1.539372
away_goals_conceded_avg_last_5    1.539372
home_points_avg_last_5            1.539372
away_points_avg_last_5            1.539372
home_overall_points_last_5        0.746004
away_overall_points_last_5        0.793369
dtype: float64

In [13]:
df_clean_for_ml = df_models.dropna(subset=features + ['target'])

In [19]:
X = df_clean_for_ml[features].astype(float)
y = df_clean_for_ml['target'].astype(int)
split_index = int(len(X) * 0.8)
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]  
y_test = y.iloc[split_index:]

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')


param_grid = {
    'n_estimators': [50, 100, 200],      
    'max_depth': [3, 5, 7],              
    'learning_rate': [0.01, 0.1]         
}

print("Starting GridSearchCV for XGBoost")


grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=tscv,                     
    scoring='accuracy',          
    n_jobs=-1,                    
    verbose=1                     
)

grid_search.fit(X_train, y_train)

best_xgb_model = grid_search.best_estimator_
predictions = best_xgb_model.predict(X_test)
final_accuracy = accuracy_score(y_test, predictions)

print(f"best params: {grid_search.best_params_}")
print(f"Hold-out Test Accuracy: {final_accuracy * 100:.2f}%")
print("="*45)

best_xgb_model = grid_search.best_estimator_

Starting GridSearchCV for XGBoost
Fitting 3 folds for each of 18 candidates, totalling 54 fits
best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
Hold-out Test Accuracy: 54.106280193236714%


In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_param_grid = {
    'n_estimators': [100, 200, 300], 
    'max_depth': [5, 10, None],      
    'min_samples_split': [2, 5, 10] 
}

rf_grid_search = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

rf_grid_search.fit(X_train, y_train)

best_rf_model = rf_grid_search.best_estimator_
rf_predictions = rf_grid_search.predict(X_test)

print(f"Best RF params: {rf_grid_search.best_params_}")
print(f"RF Hold-out Test Accuracy: {accuracy_score(y_test, rf_predictions) * 100:.2f}%")

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best RF params: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
RF Hold-out Test Accuracy: 54.35%
